In [1]:
# Load the documents 

import sys

sys.path.append("..")

from langchain_text_splitters import RecursiveCharacterTextSplitter
from src.loader import load_all_documents

documents = load_all_documents()
print("Pages loaded:", len(documents))

Pages loaded: 69


In [2]:
# How big are the pages ?

lengths = [len(doc.page_content) for doc in documents]

print("Shortest page:", min(lengths), "characters")
print("Longest page:", max(lengths), "characters")
print("Average page:", sum(lengths) // len(lengths), "characters")

Shortest page: 299 characters
Longest page: 8414 characters
Average page: 4316 characters


In [3]:
# The first split 

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

print("Pages:", len(documents))
print("Chunks:", len(chunks))
print(chunks[0].metadata)

Pages: 53
Chunks: 286
{'source': 'conditions_generales_bienvenue_2026.pdf', 'page': 2, 'title': 'Conditions générales de la convention de compte Bienvenue', 'version': 'Édition avril 2026'}


In [4]:
# Look at the chunks of page 4 

page_4_chunks = [
    chunk
    for chunk in chunks
    if chunk.metadata["source"] == "tarifs_particuliers_2026.pdf" and chunk.metadata["page"] == 4
]

for i, chunk in enumerate(page_4_chunks):
    print(f"--- Chunk {i} ({len(chunk.page_content)} characters) ---")
    print(chunk.page_content)
    print()

--- Chunk 0 (996 characters) ---
04
OUVERTURE, FONCTIONNEMENT 
ET SUIVI DE VOTRE COMPTE
  Ouverture, transformation, clôture
•  Ouverture de compte Gratuit
•  Transfert de compte dans une autre agence BNP Paribas Gratuit
•  Clôture de compte Gratuit
•   Service d’aide à la mobilité bancaire Facilimouv’  Gratuit 
(conforme aux articles L.312-1-7 et R.312-4-4 du Code monétaire et financier)  
Ce service permet un changement automatisé de la domiciliation vers votre nouveau  
compte de l’ensemble des prélèvements et virements récurrents enregistrés sur  
un compte ouvert dans une autre banque au cours des 13 derniers mois.
•   Service de changement de domiciliation bancaire Facilidom Gratuit 
Vous choisissez les émetteurs à informer de vos nouvelles coordonnées bancaires.  
Arrêt de la commercialisation le 01/01/2026 et suppression définitive du service Facilidom le 30/04/2026.
  Relevés de compte
•   Relevé standard (en ligne ou papier)  
- Mensuel Gratuit 
- Autre périodicité : décadair

In [2]:
# Reusable display function 

def show_chunks(chunks, source, page):
    selected = [
        chunk
        for chunk in chunks
        if chunk.metadata["source"] == source and chunk.metadata["page"] == page
    ]
    for i, chunk in enumerate(selected):
        print(f"--- Chunk {i} ({len(chunk.page_content)} characters) ---")
        print(chunk.page_content)
        print()

In [3]:
# The structure-aware splitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n  ", "\n•", "\n", " ", ""],
)

chunks = splitter.split_documents(documents)
print("Chunks:", len(chunks))
print()

show_chunks(chunks, "tarifs_particuliers_2026.pdf", 4)

Chunks: 334

--- Chunk 0 (854 characters) ---
04
OUVERTURE, FONCTIONNEMENT 
ET SUIVI DE VOTRE COMPTE
  Ouverture, transformation, clôture
•  Ouverture de compte Gratuit
•  Transfert de compte dans une autre agence BNP Paribas Gratuit
•  Clôture de compte Gratuit
•   Service d’aide à la mobilité bancaire Facilimouv’  Gratuit 
(conforme aux articles L.312-1-7 et R.312-4-4 du Code monétaire et financier)  
Ce service permet un changement automatisé de la domiciliation vers votre nouveau  
compte de l’ensemble des prélèvements et virements récurrents enregistrés sur  
un compte ouvert dans une autre banque au cours des 13 derniers mois.
•   Service de changement de domiciliation bancaire Facilidom Gratuit 
Vous choisissez les émetteurs à informer de vos nouvelles coordonnées bancaires.  
Arrêt de la commercialisation le 01/01/2026 et suppression définitive du service Facilidom le 30/04/2026.

--- Chunk 1 (217 characters) ---
Relevés de compte
•   Relevé standard (en ligne ou papier)  
- Me

In [7]:
from src.chunker import chunk_documents

chunks = chunk_documents(documents)
print("Chunks:", len(chunks))
print()

show_chunks(chunks, "tarifs_particuliers_2026.pdf", 4)
print("=" * 80)
show_chunks(chunks, "conditions_generales_bienvenue_2026.pdf", 5)

Chunks: 334

--- Chunk 0 (931 characters) ---
Document : Conditions et tarifs - Particuliers (Tarifs au 1er janvier 2026)

04
OUVERTURE, FONCTIONNEMENT 
ET SUIVI DE VOTRE COMPTE
  Ouverture, transformation, clôture
•  Ouverture de compte Gratuit
•  Transfert de compte dans une autre agence BNP Paribas Gratuit
•  Clôture de compte Gratuit
•   Service d’aide à la mobilité bancaire Facilimouv’  Gratuit 
(conforme aux articles L.312-1-7 et R.312-4-4 du Code monétaire et financier)  
Ce service permet un changement automatisé de la domiciliation vers votre nouveau  
compte de l’ensemble des prélèvements et virements récurrents enregistrés sur  
un compte ouvert dans une autre banque au cours des 13 derniers mois.
•   Service de changement de domiciliation bancaire Facilidom Gratuit 
Vous choisissez les émetteurs à informer de vos nouvelles coordonnées bancaires.  
Arrêt de la commercialisation le 01/01/2026 et suppression définitive du service Facilidom le 30/04/2026.

--- Chunk 1 (294 char

In [8]:
# Splitter for the agreement 

AGREEMENT_SEPARATORS = [
    r"\n(?=CHAPITRE |TITRE )",            # chapter titles
    r"\n(?=\d+(?:\.\d+)*\.?\s+[A-ZÉ])",   # numbered sections: "1. Les chèques", "1.2 Remise de chèques"
    r"\n(?= ?> )",                        # rules introduced by ">"
    r"(?<=[.;])\s",                       # end of a sentence or of a list item
    r"\s",                                # any space or line break
    r"",                                  # anywhere, as a last resort
]

agreement_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=AGREEMENT_SEPARATORS,
    is_separator_regex=True,
)

agreement_pages = [
    doc for doc in documents
    if doc.metadata["source"] == "conditions_generales_bienvenue_2026.pdf"
]
agreement_chunks = agreement_splitter.split_documents(agreement_pages)
print("Agreement chunks:", len(agreement_chunks))
print()

show_chunks(agreement_chunks, "conditions_generales_bienvenue_2026.pdf", 5)

Agreement chunks: 172

--- Chunk 0 (784 characters) ---
5
  - au moyen d’une carte de dépannage auprès des distributeurs automatiques de billets BNP Paribas (1 500 euros maximum par jour et par client) ou 
Cash Services (800 euros maximum par jour et par Client). Les modalités d’utilisation de cette carte lui seront précisées lors de sa remise en agence. 
Cette carte n’est pas acceptée par les automates de pièces de monnaie.
Le moment de réception par la Banque de l’ordre de retrait correspond à la date indiquée sur le bordereau d’opération ou le chèque remis en agence ou à 
la date de la saisie de l’ordre de retrait sur les automates de pièces de monnaie BNP Paribas ou Cash Services, ainsi que sur les distributeurs automatiques 
de billets. L’ordre de retrait d’espèces est exécuté immédiatement à compter du moment de réception.

--- Chunk 1 (923 characters) ---
> Le Client peut effectuer des versements d’espèces en euros : 
  - au guichet de l’agence disposant d’un service de caisse a

In [4]:
# Test the module 

from collections import Counter

from src.chunker import chunk_documents

chunks = chunk_documents(documents)
print("Total chunks:", len(chunks))
print(Counter(chunk.metadata["doc_type"] for chunk in chunks))
print()
print(chunks[0].metadata)
print(chunks[0].page_content[:300])

Total chunks: 343
Counter({'account_agreement': 172, 'fee_brochure': 171})

{'source': 'conditions_generales_bienvenue_2026.pdf', 'page': 2, 'title': 'Conditions générales de la convention de compte Bienvenue', 'version': 'Édition avril 2026', 'doc_type': 'account_agreement'}
Document : Conditions générales de la convention de compte Bienvenue (Édition avril 2026)

2
La convention de compte organise de manière standard la gestion du compte de dépôt en euros ou dans une devise autre que l’euro lorsque cela est prévu 
dans les conditions particulières (ci-après le « Compte


In [3]:
from collections import Counter

from src.chunker import chunk_documents

chunks = chunk_documents(documents)
print("Total chunks:", len(chunks))
print(Counter(chunk.metadata["doc_type"] for chunk in chunks))
print()

show_chunks(chunks, "conditions_fonctionnement_cartes_2026.pdf", 7)

Total chunks: 506
Counter({'account_agreement': 173, 'fee_brochure': 171, 'card_contract': 162})

--- Chunk 0 (1028 characters) ---
Document : Conditions de fonctionnement des cartes (Contrat carte) (Édition avril 2026)

7
SMONCAROSCGPDF0301
  ● un Accepteur donneur d’ordre ayant signé un contrat d’acceptation de vente à distance sécurisé avec un acquéreur adhérant au schéma de 
Cartes de paiement dont la(les) marques figure(nt) sur la Carte.
La mise en œuvre du service de réception de fonds est soumise aux conditions suivantes :
  ● BNP Paribas a contrôlé au préalable l’éligibilité de la Carte au service ;
  ● BNP Paribas a répondu favorablement à une demande d’autorisation en crédit de l’émetteur du donneur d’ordre ;
  ● BNP Paribas procède au crédit du Compte dans le délai de maximum d’un jour à réception des fonds de l’opération de transfert de fonds.
Lorsque BNP Paribas reçoit une demande d’autorisation aux fins de crédit du Compte du Titulaire bénéficiaire, il est en mesure de re

In [4]:
def content_of(chunk):
    return chunk.page_content.split("\n\n", 1)[1]


short_chunks = [chunk for chunk in chunks if len(content_of(chunk)) < 100]

print("Short chunks:", len(short_chunks))
for chunk in short_chunks:
    print(f"{chunk.metadata['source']}  p.{chunk.metadata['page']}  {repr(content_of(chunk))}")

Short chunks: 25
conditions_fonctionnement_cartes_2026.pdf  p.4  'IV – FORME DU CONSENTEMENT ET IRRÉVOCABILITÉ'
conditions_fonctionnement_cartes_2026.pdf  p.5  '5\nSMONCAROSCGPDF0301'
conditions_fonctionnement_cartes_2026.pdf  p.8  '8\nSMONCAROSCGPDF0301'
conditions_fonctionnement_cartes_2026.pdf  p.9  '9\nSMONCAROSCGPDF0301'
conditions_fonctionnement_cartes_2026.pdf  p.9  'XVII – REMBOURSEMENT DES OPÉRATIONS NON AUTORISÉES OU MAL EXÉCUTÉES'
conditions_fonctionnement_cartes_2026.pdf  p.10  '10\nSMONCAROSCGPDF0301'
conditions_fonctionnement_cartes_2026.pdf  p.10  'XVIII – DONNÉES PERSONNELLES ET SECRET BANCAIRE'
conditions_fonctionnement_cartes_2026.pdf  p.11  'XIX – CONDITIONS FINANCIÈRES'
conditions_fonctionnement_cartes_2026.pdf  p.14  '14\nSMONCAROSCGPDF0301\n3 – Traitement de données personnelles propres au Schéma CB'
conditions_generales_bienvenue_2026.pdf  p.3  'CHAPITRE II - LE FONCTIONNEMENT DU COMPTE\n1. Types de compte proposés'
conditions_generales_bienvenue_2026.pdf  p.4  '

In [3]:
from collections import Counter

from src.chunker import chunk_documents

chunks = chunk_documents(documents)
print("Total chunks:", len(chunks))
print(Counter(chunk.metadata["doc_type"] for chunk in chunks))


def content_of(chunk):
    return chunk.page_content.split("\n\n", 1)[1]


short_chunks = [chunk for chunk in chunks if len(content_of(chunk)) < 100]
print("Short chunks:", len(short_chunks))
print()

for chunk in chunks:
    if "XVII – REMBOURSEMENT" in chunk.page_content:
        print(chunk.page_content[:600])

Total chunks: 481
Counter({'fee_brochure': 166, 'account_agreement': 162, 'card_contract': 153})
Short chunks: 0

Document : Conditions de fonctionnement des cartes (Contrat carte) (Édition avril 2026)

XVII – REMBOURSEMENT DES OPÉRATIONS NON AUTORISÉES OU MAL EXÉCUTÉES
17.1 Opération de paiement non autorisée
Le Titulaire de la Carte et/ou du Compte, est remboursé immédiatement et au plus tard, le premier jour ouvrable suivant la réception de la 
contestation de l’opération :
  ● du montant de l’opération contestée de bonne foi par le Titulaire de la Carte dans le cas de perte et/ou vol, d’utilisation frauduleuse et/ou de 
détournement de sa Carte et des données qui y sont liées, survenue avant la demand
